In [1]:
! pip install pytorch-tabular rtdl-revisited optuna scikit-learn pandas numpy torch

ERROR: Could not find a version that satisfies the requirement rtdl-revisited (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Asus\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for rtdl-revisited


In [5]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'deep_tabular_results' not in globals():
    deep_tabular_results = []

print(f"PyTorch Sürümü: {torch.__version__} | Aktif Hesaplama Cihazı: {device}")
print("Gorishniy ResNet, Yandex NODE & FT-Transformer, SAINT ve DANNet/SAIL Motorları Hazır.")

PyTorch Sürümü: 2.5.1+cpu | Aktif Hesaplama Cihazı: cpu
Gorishniy ResNet, Yandex NODE & FT-Transformer, SAINT ve DANNet/SAIL Motorları Hazır.


In [4]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)
df.describe()

df.head(10)

Veri Boyutu: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [6]:
X_raw = df.drop(columns=["area"])
y_raw = df["area"].values

X_proc = X_raw.copy()
for col in ["month", "day"]:
    if col in X_proc.columns:
        le = LabelEncoder()
        X_proc[col] = le.fit_transform(X_proc[col])

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_proc, y_raw, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_test_scaled = scaler.transform(X_test_df)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

if 'deep_tabular_results' not in globals():
    deep_tabular_results = []

def evaluate_deep_tabular(y_true, y_pred_ha, stage_name, model_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    deep_tabular_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": model_name,
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[{model_name.upper()}] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_current = pd.DataFrame(deep_tabular_results)
    styled_table = (
        df_current.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

print(f"X_train_scaled Boyutu: {X_train_scaled.shape} | X_test_scaled Boyutu: {X_test_scaled.shape}")
print(f"y_train Medyan: {np.median(y_train):.4f} ha | y_test Medyan: {np.median(y_test):.4f} ha")
print(f"y_train_log Medyan: {np.median(y_train_log):.4f} | y_test_log Medyan: {np.median(y_test_log):.4f}")

X_train_scaled Boyutu: (413, 12) | X_test_scaled Boyutu: (104, 12)
y_train Medyan: 0.5400 ha | y_test Medyan: 0.4950 ha
y_train_log Medyan: 0.4318 | y_test_log Medyan: 0.4020


In [7]:
print("=========================================================================================")
print("UÇ YANGINLARIN (EXTREME FIRES) TRAIN VE TEST SETİNDEKİ DAĞILIM KONTROLÜ")
print("=========================================================================================")
print(f"y_train -> Maksimum: {y_train.max():.2f} ha | >100 ha Sayısı: {(y_train > 100).sum()} | >50 ha Sayısı: {(y_train > 50).sum()}")
print(f"y_test  -> Maksimum: {y_test.max():.2f} ha  | >100 ha Sayısı: {(y_test > 100).sum()}  | >50 ha Sayısı: {(y_test > 50).sum()}")
print("-----------------------------------------------------------------------------------------")
print(f"y_train Ortalama: {y_train.mean():.2f} ha | Standart Sapma: {y_train.std():.2f} ha")
print(f"y_test  Ortalama: {y_test.mean():.2f} ha  | Standart Sapma: {y_test.std():.2f} ha")
print("=========================================================================================")

UÇ YANGINLARIN (EXTREME FIRES) TRAIN VE TEST SETİNDEKİ DAĞILIM KONTROLÜ
y_train -> Maksimum: 746.28 ha | >100 ha Sayısı: 9 | >50 ha Sayısı: 18
y_test  -> Maksimum: 1090.84 ha  | >100 ha Sayısı: 2  | >50 ha Sayısı: 6
-----------------------------------------------------------------------------------------
y_train Ortalama: 11.13 ha | Standart Sapma: 45.60 ha
y_test  Ortalama: 19.66 ha  | Standart Sapma: 108.57 ha


## NODE Deneyleri:

### Deney 1:

In [8]:
class NODETabularLayer(nn.Module):
    def __init__(self, num_features, num_trees=16, depth=4):
        super().__init__()
        self.num_trees = num_trees
        self.depth = depth
        self.feature_weights = nn.Parameter(torch.randn(num_trees, depth, num_features) * 0.1)
        self.thresholds = nn.Parameter(torch.randn(num_trees, depth) * 0.1)
        self.scales = nn.Parameter(torch.ones(num_trees, depth))
        self.response = nn.Parameter(torch.randn(num_trees, 2 ** depth) * 0.1)

    def forward(self, x):
        batch_size = x.size(0)
        feat_sel = torch.softmax(self.feature_weights, dim=-1)
        sel_vals = torch.einsum('bf,tdf->btd', x, feat_sel)
        diff = (sel_vals - self.thresholds.unsqueeze(0)) * self.scales.unsqueeze(0)
        prob_right = torch.sigmoid(diff)
        prob_left = 1.0 - prob_right
        
        tree_probs = torch.ones(batch_size, self.num_trees, 1, device=x.device)
        for d in range(self.depth):
            pr = prob_right[:, :, d].unsqueeze(-1)
            pl = prob_left[:, :, d].unsqueeze(-1)
            tree_probs = torch.cat([tree_probs * pl, tree_probs * pr], dim=-1)
            
        out = torch.einsum('btl,tl->b', tree_probs, self.response)
        return out

class NODETabular(nn.Module):
    def __init__(self, num_features, num_layers=2, num_trees=16, depth=4):
        super().__init__()
        self.layers = nn.ModuleList([NODETabularLayer(num_features if i == 0 else num_features + i, num_trees, depth) for i in range(num_layers)])
        self.final_proj = nn.Linear(num_features + num_layers, 1)

    def forward(self, x):
        curr_x = x
        for layer in self.layers:
            out_l = layer(curr_x).unsqueeze(-1)
            curr_x = torch.cat([curr_x, out_l], dim=-1)
        return self.final_proj(curr_x).squeeze(-1)

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_node_raw = TensorDataset(X_tr_t, y_tr_t)
loader_node_raw = DataLoader(dataset_node_raw, batch_size=32, shuffle=True)

model_node_def = NODETabular(num_features=X_train_scaled.shape[1], num_layers=2, num_trees=16, depth=4)
optimizer_node = optim.Adam(model_node_def.parameters(), lr=0.005)
criterion_node = nn.MSELoss()

model_node_def.train()
for epoch in range(150):
    for bx, by in loader_node_raw:
        optimizer_node.zero_grad()
        preds_b = model_node_def(bx)
        loss = criterion_node(preds_b, by)
        loss.backward()
        optimizer_node.step()

model_node_def.eval()
with torch.no_grad():
    y_pred_node_raw = model_node_def(X_te_t).numpy()

evaluate_deep_tabular(y_test, y_pred_node_raw, "1 - Normal Veri (Varsayılan)", "NODE")

[NODE] - 1 - NORMAL VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 10.8458 ha | MAE: 24.0999 ha | RMSE: 108.8814 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814


(np.float64(10.845820903778076),
 24.099863693163943,
 np.float64(108.88136665450759))

### Deney 2:

In [10]:
def objective_node_raw(trial):
    params = {
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "num_trees": trial.suggest_categorical("num_trees", [8, 16, 32]),
        "depth": trial.suggest_int("depth", 3, 5),
        "lr": trial.suggest_float("lr", 0.001, 0.02, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = NODETabular(num_features=X_train_scaled.shape[1], num_layers=params["num_layers"], num_trees=params["num_trees"], depth=params["depth"])
        opt = optim.Adam(model.parameters(), lr=params["lr"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_val = model(X_val_t).numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_val)))
        
    return np.mean(cv_mads)

study_node_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_node_raw.optimize(objective_node_raw, n_trials=15, show_progress_bar=False)
best_node_p_raw = study_node_raw.best_params

model_node_opt_raw = NODETabular(num_features=X_train_scaled.shape[1], num_layers=best_node_p_raw["num_layers"], num_trees=best_node_p_raw["num_trees"], depth=best_node_p_raw["depth"])
opt_node_raw = optim.Adam(model_node_opt_raw.parameters(), lr=best_node_p_raw["lr"])
crit_node_raw = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=32, shuffle=True)

model_node_opt_raw.train()
for epoch in range(120):
    for bx, by in ld_full_tr:
        opt_node_raw.zero_grad()
        p_b = model_node_opt_raw(bx)
        l = crit_node_raw(p_b, by)
        l.backward()
        opt_node_raw.step()

model_node_opt_raw.eval()
with torch.no_grad():
    y_pred_node_opt_raw = model_node_opt_raw(X_te_t).numpy()

evaluate_deep_tabular(y_test, y_pred_node_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)", "NODE")

[NODE] - 2 - NORMAL VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 10.9387 ha | MAE: 24.1425 ha | RMSE: 108.8989 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
1,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
2,2 - Normal Veri (Bayesyen Optuna HPO),NODE,10.9387,24.1425,108.8989


(np.float64(10.93868350982666),
 24.142502902837897,
 np.float64(108.89891563797767))

### Deney 3:

In [11]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_node_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_node_log = DataLoader(dataset_node_log, batch_size=32, shuffle=True)

model_node_log_def = NODETabular(num_features=X_train_scaled.shape[1], num_layers=2, num_trees=16, depth=4)
optimizer_node_log = optim.Adam(model_node_log_def.parameters(), lr=0.005)
criterion_node_log = nn.MSELoss()

model_node_log_def.train()
for epoch in range(150):
    for bx, by in loader_node_log:
        optimizer_node_log.zero_grad()
        preds_b = model_node_log_def(bx)
        loss = criterion_node_log(preds_b, by)
        loss.backward()
        optimizer_node_log.step()

model_node_log_def.eval()
with torch.no_grad():
    preds_log_val = model_node_log_def(X_te_t).numpy()

y_pred_node_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_deep_tabular(y_test, y_pred_node_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)", "NODE")

[NODE] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 2.0722 ha | MAE: 19.8144 ha | RMSE: 109.9858 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
1,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
2,2 - Normal Veri (Bayesyen Optuna HPO),NODE,10.9387,24.1425,108.8989
3,3 - Log-Dönüşümlü Veri (Varsayılan),NODE,2.0722,19.8144,109.9858


(np.float64(2.0722079277038574),
 19.814380882519938,
 np.float64(109.98575533391329))

### Deney 4:

In [13]:
if len(deep_tabular_results) > 0 and deep_tabular_results[-1]["Deney Aşaması"] == "4 - Log Veri (Bayesyen Optuna HPO)":
    deep_tabular_results.pop()

def objective_node_log_v2(trial):
    params = {
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "num_trees": trial.suggest_categorical("num_trees", [12, 16, 24, 32]),
        "depth": trial.suggest_int("depth", 3, 5),
        "lr": trial.suggest_float("lr", 0.001, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = NODETabular(num_features=X_train_scaled.shape[1], num_layers=params["num_layers"], num_trees=params["num_trees"], depth=params["depth"])
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(120):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_log_val = model(X_val_t).numpy()
        preds_ha_val = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_val)))
        
    return np.mean(cv_mads)

study_node_log_v2 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_node_log_v2.optimize(objective_node_log_v2, n_trials=25, show_progress_bar=False)
best_node_p_v2 = study_node_log_v2.best_params

model_node_opt_v2 = NODETabular(
    num_features=X_train_scaled.shape[1],
    num_layers=best_node_p_v2["num_layers"],
    num_trees=best_node_p_v2["num_trees"],
    depth=best_node_p_v2["depth"]
)
opt_node_v2 = optim.AdamW(model_node_opt_v2.parameters(), lr=best_node_p_v2["lr"], weight_decay=best_node_p_v2["weight_decay"])
scheduler_node_v2 = optim.lr_scheduler.CosineAnnealingLR(opt_node_v2, T_max=150)
crit_node_v2 = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=32, shuffle=True)

model_node_opt_v2.train()
for epoch in range(150):
    for bx, by in ld_full_tr_log:
        opt_node_v2.zero_grad()
        p_b = model_node_opt_v2(bx)
        l = crit_node_v2(p_b, by)
        l.backward()
        opt_node_v2.step()
    scheduler_node_v2.step()

model_node_opt_v2.eval()
with torch.no_grad():
    preds_log_opt_v2 = model_node_opt_v2(X_te_t).numpy()

y_pred_node_opt_v2_ha = np.clip(np.expm1(preds_log_opt_v2), 0, None)

evaluate_deep_tabular(y_test, y_pred_node_opt_v2_ha, "4 - Log Veri (Bayesyen Optuna HPO)", "NODE")

[NODE] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 2.0192 ha | MAE: 19.8150 ha | RMSE: 109.9913 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
1,1 - Normal Veri (Varsayılan),NODE,10.8458,24.0999,108.8814
2,2 - Normal Veri (Bayesyen Optuna HPO),NODE,10.9387,24.1425,108.8989
3,3 - Log-Dönüşümlü Veri (Varsayılan),NODE,2.0722,19.8144,109.9858
4,4 - Log Veri (Bayesyen Optuna HPO),NODE,2.0192,19.8150,109.9913


(np.float64(2.019194722175598),
 19.81496590669338,
 np.float64(109.99133701963228))

## Tabular Res-Net Baseline Deneyleri:

### Deney 1:

In [15]:
class TabularResNetBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return x + self.block(x)

class TabularResNet(nn.Module):
    def __init__(self, num_features, hidden_dim=64, num_blocks=2, dropout=0.1):
        super().__init__()
        self.input_layer = nn.Linear(num_features, hidden_dim)
        self.blocks = nn.ModuleList([TabularResNetBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        h = self.input_layer(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h).squeeze(-1)

if 'resnet_results' not in globals() or len(resnet_results) > 4:
    resnet_results = []

def evaluate_resnet(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    resnet_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "ResNet Baseline",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[RESNET BASELINE] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_resnet = pd.DataFrame(resnet_results)
    styled_table = (
        df_resnet.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Blues")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#1f77b4'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1f77b4'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_resnet_raw = TensorDataset(X_tr_t, y_tr_t)
loader_resnet_raw = DataLoader(dataset_resnet_raw, batch_size=32, shuffle=True)

model_resnet_def = TabularResNet(num_features=X_train_scaled.shape[1], hidden_dim=64, num_blocks=2, dropout=0.1)
optimizer_resnet = optim.Adam(model_resnet_def.parameters(), lr=0.001)
criterion_resnet = nn.MSELoss()

model_resnet_def.train()
for epoch in range(150):
    for bx, by in loader_resnet_raw:
        optimizer_resnet.zero_grad()
        preds_b = model_resnet_def(bx)
        loss = criterion_resnet(preds_b, by)
        loss.backward()
        optimizer_resnet.step()

model_resnet_def.eval()
with torch.no_grad():
    y_pred_resnet_raw = model_resnet_def(X_te_t).numpy()

evaluate_resnet(y_test, y_pred_resnet_raw, "1 - Normal Veri (Varsayılan)")

[RESNET BASELINE] - 1 - NORMAL VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 10.6147 ha | MAE: 23.9659 ha | RMSE: 108.9358 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),ResNet Baseline,10.6147,23.9659,108.9358


(np.float64(10.614718914031982),
 23.965939813760606,
 np.float64(108.9357691340372))

### Deney 2:

In [16]:
def objective_resnet_raw(trial):
    params = {
        "num_blocks": trial.suggest_int("num_blocks", 1, 4),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.01, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = TabularResNet(
            num_features=X_train_scaled.shape[1],
            hidden_dim=params["hidden_dim"],
            num_blocks=params["num_blocks"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_val = model(X_val_t).numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_val)))
        
    return np.mean(cv_mads)

study_resnet_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_resnet_raw.optimize(objective_resnet_raw, n_trials=15, show_progress_bar=False)
best_resnet_p_raw = study_resnet_raw.best_params

model_resnet_opt_raw = TabularResNet(
    num_features=X_train_scaled.shape[1],
    hidden_dim=best_resnet_p_raw["hidden_dim"],
    num_blocks=best_resnet_p_raw["num_blocks"],
    dropout=best_resnet_p_raw["dropout"]
)
opt_resnet_raw = optim.AdamW(model_resnet_opt_raw.parameters(), lr=best_resnet_p_raw["lr"], weight_decay=best_resnet_p_raw["weight_decay"])
scheduler_resnet_raw = optim.lr_scheduler.CosineAnnealingLR(opt_resnet_raw, T_max=150)
crit_resnet_raw = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=32, shuffle=True)

model_resnet_opt_raw.train()
for epoch in range(150):
    for bx, by in ld_full_tr:
        opt_resnet_raw.zero_grad()
        p_b = model_resnet_opt_raw(bx)
        l = crit_resnet_raw(p_b, by)
        l.backward()
        opt_resnet_raw.step()
    scheduler_resnet_raw.step()

model_resnet_opt_raw.eval()
with torch.no_grad():
    y_pred_resnet_opt_raw = model_resnet_opt_raw(X_te_t).numpy()

evaluate_resnet(y_test, y_pred_resnet_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)")

[RESNET BASELINE] - 2 - NORMAL VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 9.1150 ha | MAE: 23.0454 ha | RMSE: 109.0552 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),ResNet Baseline,10.6147,23.9659,108.9358
1,2 - Normal Veri (Bayesyen Optuna HPO),ResNet Baseline,9.1150,23.0454,109.0552


(np.float64(9.115026473999023),
 23.045426326531626,
 np.float64(109.05520307962571))

### Deney 3:

In [17]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_resnet_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_resnet_log = DataLoader(dataset_resnet_log, batch_size=32, shuffle=True)

model_resnet_log_def = TabularResNet(num_features=X_train_scaled.shape[1], hidden_dim=64, num_blocks=2, dropout=0.1)
optimizer_resnet_log = optim.Adam(model_resnet_log_def.parameters(), lr=0.001)
criterion_resnet_log = nn.MSELoss()

model_resnet_log_def.train()
for epoch in range(150):
    for bx, by in loader_resnet_log:
        optimizer_resnet_log.zero_grad()
        preds_b = model_resnet_log_def(bx)
        loss = criterion_resnet_log(preds_b, by)
        loss.backward()
        optimizer_resnet_log.step()

model_resnet_log_def.eval()
with torch.no_grad():
    preds_log_val = model_resnet_log_def(X_te_t).numpy()

y_pred_resnet_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_resnet(y_test, y_pred_resnet_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)")

[RESNET BASELINE] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 1.9859 ha | MAE: 19.7990 ha | RMSE: 109.9965 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),ResNet Baseline,10.6147,23.9659,108.9358
1,2 - Normal Veri (Bayesyen Optuna HPO),ResNet Baseline,9.1150,23.0454,109.0552
2,3 - Log-Dönüşümlü Veri (Varsayılan),ResNet Baseline,1.9859,19.7990,109.9965


(np.float64(1.98592209815979),
 19.79902393515293,
 np.float64(109.99652061131118))

### Deney 4:

In [18]:
def objective_resnet_log(trial):
    params = {
        "num_blocks": trial.suggest_int("num_blocks", 1, 4),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = TabularResNet(
            num_features=X_train_scaled.shape[1],
            hidden_dim=params["hidden_dim"],
            num_blocks=params["num_blocks"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(120):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_log_val = model(X_val_t).numpy()
        preds_ha_val = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_val)))
        
    return np.mean(cv_mads)

study_resnet_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_resnet_log.optimize(objective_resnet_log, n_trials=25, show_progress_bar=False)
best_resnet_p_log = study_resnet_log.best_params

model_resnet_opt_log = TabularResNet(
    num_features=X_train_scaled.shape[1],
    hidden_dim=best_resnet_p_log["hidden_dim"],
    num_blocks=best_resnet_p_log["num_blocks"],
    dropout=best_resnet_p_log["dropout"]
)
opt_resnet_log = optim.AdamW(model_resnet_opt_log.parameters(), lr=best_resnet_p_log["lr"], weight_decay=best_resnet_p_log["weight_decay"])
scheduler_resnet_log = optim.lr_scheduler.CosineAnnealingLR(opt_resnet_log, T_max=150)
crit_resnet_log = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=32, shuffle=True)

model_resnet_opt_log.train()
for epoch in range(150):
    for bx, by in ld_full_tr_log:
        opt_resnet_log.zero_grad()
        p_b = model_resnet_opt_log(bx)
        l = crit_resnet_log(p_b, by)
        l.backward()
        opt_resnet_log.step()
    scheduler_resnet_log.step()

model_resnet_opt_log.eval()
with torch.no_grad():
    preds_log_opt_resnet = model_resnet_opt_log(X_te_t).numpy()

y_pred_resnet_opt_log_ha = np.clip(np.expm1(preds_log_opt_resnet), 0, None)

evaluate_resnet(y_test, y_pred_resnet_opt_log_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

[RESNET BASELINE] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 1.8805 ha | MAE: 19.8016 ha | RMSE: 110.0257 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),ResNet Baseline,10.6147,23.9659,108.9358
1,2 - Normal Veri (Bayesyen Optuna HPO),ResNet Baseline,9.1150,23.0454,109.0552
2,3 - Log-Dönüşümlü Veri (Varsayılan),ResNet Baseline,1.9859,19.7990,109.9965
3,4 - Log Veri (Bayesyen Optuna HPO),ResNet Baseline,1.8805,19.8016,110.0257


(np.float64(1.8804881572723389),
 19.80160252149288,
 np.float64(110.02572238635152))

## FT-Transformer Deneyleri:

### Deney 1:

In [19]:
class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, d_token=32):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(num_features, d_token) * 0.02)
        self.biases = nn.Parameter(torch.randn(num_features, d_token) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

    def forward(self, x):
        batch_size = x.size(0)
        tokens = x.unsqueeze(-1) * self.weights.unsqueeze(0) + self.biases.unsqueeze(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        return torch.cat([cls_tokens, tokens], dim=1)

class FTTransformerTabular(nn.Module):
    def __init__(self, num_features, d_token=32, n_blocks=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_token)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*2, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.ReLU(),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):
        tokens = self.tokenizer(x)
        out = self.transformer(tokens)
        return self.head(out[:, 0, :]).squeeze(-1)

if 'ft_results' not in globals() or len(ft_results) > 4:
    ft_results = []

def evaluate_ft(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    ft_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "FT-Transformer",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[FT-TRANSFORMER] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_ft = pd.DataFrame(ft_results)
    styled_table = (
        df_ft.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Purples")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#4a148c'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#4a148c'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_ft_raw = TensorDataset(X_tr_t, y_tr_t)
loader_ft_raw = DataLoader(dataset_ft_raw, batch_size=32, shuffle=True)

model_ft_def = FTTransformerTabular(num_features=X_train_scaled.shape[1], d_token=32, n_blocks=2, n_heads=4, dropout=0.1)
optimizer_ft = optim.AdamW(model_ft_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_ft = nn.MSELoss()

model_ft_def.train()
for epoch in range(150):
    for bx, by in loader_ft_raw:
        optimizer_ft.zero_grad()
        preds_b = model_ft_def(bx)
        loss = criterion_ft(preds_b, by)
        loss.backward()
        optimizer_ft.step()

model_ft_def.eval()
with torch.no_grad():
    y_pred_ft_raw = model_ft_def(X_te_t).numpy()

evaluate_ft(y_test, y_pred_ft_raw, "1 - Normal Veri (Varsayılan)")

[FT-TRANSFORMER] - 1 - NORMAL VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 11.1170 ha | MAE: 24.1742 ha | RMSE: 108.9074 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),FT-Transformer,11.1170,24.1742,108.9074


(np.float64(11.116989612579346),
 24.17424091815948,
 np.float64(108.90739895139998))

### Deney 2:

In [20]:
def objective_ft_raw(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32, 64])
    n_blk = trial.suggest_int("n_blocks", 1, 3)
    n_hds = trial.suggest_categorical("n_heads", [2, 4, 8])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_blocks": n_blk,
        "n_heads": n_hds,
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = FTTransformerTabular(
            num_features=X_train_scaled.shape[1],
            d_token=params["d_token"],
            n_blocks=params["n_blocks"],
            n_heads=params["n_heads"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_val = model(X_val_t).numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_val)))
        
    return np.mean(cv_mads)

study_ft_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_ft_raw.optimize(objective_ft_raw, n_trials=15, show_progress_bar=False)
best_ft_p_raw = study_ft_raw.best_params

if best_ft_p_raw["d_token"] % best_ft_p_raw["n_heads"] != 0:
    best_ft_p_raw["n_heads"] = 2

model_ft_opt_raw = FTTransformerTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_ft_p_raw["d_token"],
    n_blocks=best_ft_p_raw["n_blocks"],
    n_heads=best_ft_p_raw["n_heads"],
    dropout=best_ft_p_raw["dropout"]
)
opt_ft_raw = optim.AdamW(model_ft_opt_raw.parameters(), lr=best_ft_p_raw["lr"], weight_decay=best_ft_p_raw["weight_decay"])
scheduler_ft_raw = optim.lr_scheduler.CosineAnnealingLR(opt_ft_raw, T_max=150)
crit_ft_raw = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=32, shuffle=True)

model_ft_opt_raw.train()
for epoch in range(150):
    for bx, by in ld_full_tr:
        opt_ft_raw.zero_grad()
        p_b = model_ft_opt_raw(bx)
        l = crit_ft_raw(p_b, by)
        l.backward()
        opt_ft_raw.step()
    scheduler_ft_raw.step()

model_ft_opt_raw.eval()
with torch.no_grad():
    y_pred_ft_opt_raw = model_ft_opt_raw(X_te_t).numpy()

evaluate_ft(y_test, y_pred_ft_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)")

[FT-TRANSFORMER] - 2 - NORMAL VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 7.3929 ha | MAE: 21.9608 ha | RMSE: 109.2621 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),FT-Transformer,11.1170,24.1742,108.9074
1,2 - Normal Veri (Bayesyen Optuna HPO),FT-Transformer,7.3929,21.9608,109.2621


(np.float64(7.392919301986694),
 21.96084155156062,
 np.float64(109.26207971015425))

### Deney 3:

In [21]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_ft_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_ft_log = DataLoader(dataset_ft_log, batch_size=32, shuffle=True)

model_ft_log_def = FTTransformerTabular(num_features=X_train_scaled.shape[1], d_token=32, n_blocks=2, n_heads=4, dropout=0.1)
optimizer_ft_log = optim.AdamW(model_ft_log_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_ft_log = nn.MSELoss()

model_ft_log_def.train()
for epoch in range(150):
    for bx, by in loader_ft_log:
        optimizer_ft_log.zero_grad()
        preds_b = model_ft_log_def(bx)
        loss = criterion_ft_log(preds_b, by)
        loss.backward()
        optimizer_ft_log.step()

model_ft_log_def.eval()
with torch.no_grad():
    preds_log_val = model_ft_log_def(X_te_t).numpy()

y_pred_ft_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_ft(y_test, y_pred_ft_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)")

[FT-TRANSFORMER] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 2.0676 ha | MAE: 19.8294 ha | RMSE: 109.9873 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),FT-Transformer,11.1170,24.1742,108.9074
1,2 - Normal Veri (Bayesyen Optuna HPO),FT-Transformer,7.3929,21.9608,109.2621
2,3 - Log-Dönüşümlü Veri (Varsayılan),FT-Transformer,2.0676,19.8294,109.9873


(np.float64(2.067595601081848),
 19.829386952473563,
 np.float64(109.98726729948334))

### Deney 4:

In [22]:
def objective_ft_log(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32, 64])
    n_blk = trial.suggest_int("n_blocks", 1, 3)
    n_hds = trial.suggest_categorical("n_heads", [2, 4, 8])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_blocks": n_blk,
        "n_heads": n_hds,
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = FTTransformerTabular(
            num_features=X_train_scaled.shape[1],
            d_token=params["d_token"],
            n_blocks=params["n_blocks"],
            n_heads=params["n_heads"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(120):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_log_val = model(X_val_t).numpy()
        preds_ha_val = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_val)))
        
    return np.mean(cv_mads)

study_ft_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_ft_log.optimize(objective_ft_log, n_trials=25, show_progress_bar=False)
best_ft_p_log = study_ft_log.best_params

if best_ft_p_log["d_token"] % best_ft_p_log["n_heads"] != 0:
    best_ft_p_log["n_heads"] = 2

model_ft_opt_log = FTTransformerTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_ft_p_log["d_token"],
    n_blocks=best_ft_p_log["n_blocks"],
    n_heads=best_ft_p_log["n_heads"],
    dropout=best_ft_p_log["dropout"]
)
opt_ft_log = optim.AdamW(model_ft_opt_log.parameters(), lr=best_ft_p_log["lr"], weight_decay=best_ft_p_log["weight_decay"])
scheduler_ft_log = optim.lr_scheduler.CosineAnnealingLR(opt_ft_log, T_max=150)
crit_ft_log = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=32, shuffle=True)

model_ft_opt_log.train()
for epoch in range(150):
    for bx, by in ld_full_tr_log:
        opt_ft_log.zero_grad()
        p_b = model_ft_opt_log(bx)
        l = crit_ft_log(p_b, by)
        l.backward()
        opt_ft_log.step()
    scheduler_ft_log.step()

model_ft_opt_log.eval()
with torch.no_grad():
    preds_log_opt_ft = model_ft_opt_log(X_te_t).numpy()

y_pred_ft_opt_log_ha = np.clip(np.expm1(preds_log_opt_ft), 0, None)

evaluate_ft(y_test, y_pred_ft_opt_log_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

[FT-TRANSFORMER] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 2.0683 ha | MAE: 19.8300 ha | RMSE: 109.9872 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),FT-Transformer,11.1170,24.1742,108.9074
1,2 - Normal Veri (Bayesyen Optuna HPO),FT-Transformer,7.3929,21.9608,109.2621
2,3 - Log-Dönüşümlü Veri (Varsayılan),FT-Transformer,2.0676,19.8294,109.9873
3,4 - Log Veri (Bayesyen Optuna HPO),FT-Transformer,2.0683,19.8300,109.9872


(np.float64(2.0683023929595947),
 19.83003806215066,
 np.float64(109.98716824983741))

## SAINT Deneyleri:

### Deney 1:

In [23]:
class SAINTTabular(nn.Module):
    def __init__(self, num_features, d_token=32, n_blocks=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_token)
        self.col_encoder = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*2, dropout=dropout, batch_first=True
        )
        self.row_encoder = nn.TransformerEncoderLayer(
            d_model=num_features+1, nhead=1 if (num_features+1)%n_heads!=0 else n_heads, dim_feedforward=(num_features+1)*2, dropout=dropout, batch_first=True
        )
        self.n_blocks = n_blocks
        self.head = nn.Linear(d_token, 1)

    def forward(self, x):
        tokens = self.tokenizer(x)
        for _ in range(self.n_blocks):
            tokens = self.col_encoder(tokens)
            row_tokens = tokens.permute(0, 2, 1)
            row_tokens = self.row_encoder(row_tokens)
            tokens = row_tokens.permute(0, 2, 1)
        return self.head(tokens[:, 0, :]).squeeze(-1)

if 'saint_results' not in globals() or len(saint_results) > 4:
    saint_results = []

def evaluate_saint(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    saint_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "SAINT Transformer",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    
    df_saint = pd.DataFrame(saint_results)
    styled_table = (
        df_saint.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Greens")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#1b5e20'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1b5e20'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_saint_raw = TensorDataset(X_tr_t, y_tr_t)
loader_saint_raw = DataLoader(dataset_saint_raw, batch_size=32, shuffle=True)

model_saint_def = SAINTTabular(num_features=X_train_scaled.shape[1], d_token=32, n_blocks=2, n_heads=4, dropout=0.1)
optimizer_saint = optim.AdamW(model_saint_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_saint = nn.MSELoss()

model_saint_def.train()
for epoch in range(150):
    for bx, by in loader_saint_raw:
        optimizer_saint.zero_grad()
        preds_b = model_saint_def(bx)
        loss = criterion_saint(preds_b, by)
        loss.backward()
        optimizer_saint.step()

model_saint_def.eval()
with torch.no_grad():
    y_pred_saint_raw = model_saint_def(X_te_t).numpy()

evaluate_saint(y_test, y_pred_saint_raw, "1 - Normal Veri (Varsayılan)")

Model -> MAD: 11.4767 ha | MAE: 24.4245 ha | RMSE: 108.8721 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAINT Transformer,11.4767,24.4245,108.8721


(np.float64(11.47672986984253),
 24.42450606712928,
 np.float64(108.87210360860792))

### Deney 2:

In [24]:
def objective_saint_raw(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32, 64])
    n_blk = trial.suggest_int("n_blocks", 1, 3)
    n_hds = trial.suggest_categorical("n_heads", [2, 4, 8])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_blocks": n_blk,
        "n_heads": n_hds,
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = SAINTTabular(
            num_features=X_train_scaled.shape[1],
            d_token=params["d_token"],
            n_blocks=params["n_blocks"],
            n_heads=params["n_heads"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.MSELoss()
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            preds_val = model(X_val_t).numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_val)))
        
    return np.mean(cv_mads)

study_saint_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_saint_raw.optimize(objective_saint_raw, n_trials=15, show_progress_bar=False)
best_saint_p_raw = study_saint_raw.best_params

if best_saint_p_raw["d_token"] % best_saint_p_raw["n_heads"] != 0:
    best_saint_p_raw["n_heads"] = 2

model_saint_opt_raw = SAINTTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_saint_p_raw["d_token"],
    n_blocks=best_saint_p_raw["n_blocks"],
    n_heads=best_saint_p_raw["n_heads"],
    dropout=best_saint_p_raw["dropout"]
)
opt_saint_raw = optim.AdamW(model_saint_opt_raw.parameters(), lr=best_saint_p_raw["lr"], weight_decay=best_saint_p_raw["weight_decay"])
scheduler_saint_raw = optim.lr_scheduler.CosineAnnealingLR(opt_saint_raw, T_max=150)
crit_saint_raw = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=32, shuffle=True)

model_saint_opt_raw.train()
for epoch in range(150):
    for bx, by in ld_full_tr:
        opt_saint_raw.zero_grad()
        p_b = model_saint_opt_raw(bx)
        l = crit_saint_raw(p_b, by)
        l.backward()
        opt_saint_raw.step()
    scheduler_saint_raw.step()

model_saint_opt_raw.eval()
with torch.no_grad():
    y_pred_saint_opt_raw = model_saint_opt_raw(X_te_t).numpy()

evaluate_saint(y_test, y_pred_saint_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)")

Model -> MAD: 11.0679 ha | MAE: 24.1684 ha | RMSE: 108.9126 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAINT Transformer,11.4767,24.4245,108.8721
1,2 - Normal Veri (Bayesyen Optuna HPO),SAINT Transformer,11.0679,24.1684,108.9126


(np.float64(11.067915916442871),
 24.168355005337638,
 np.float64(108.9126447296469))

### Deney 3:

In [25]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_saint_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_saint_log = DataLoader(dataset_saint_log, batch_size=32, shuffle=True)

model_saint_log_def = SAINTTabular(num_features=X_train_scaled.shape[1], d_token=32, n_blocks=2, n_heads=4, dropout=0.1)
optimizer_saint_log = optim.AdamW(model_saint_log_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_saint_log = nn.MSELoss()

model_saint_log_def.train()
for epoch in range(150):
    for bx, by in loader_saint_log:
        optimizer_saint_log.zero_grad()
        preds_b = model_saint_log_def(bx)
        loss = criterion_saint_log(preds_b, by)
        loss.backward()
        optimizer_saint_log.step()

model_saint_log_def.eval()
with torch.no_grad():
    preds_log_val = model_saint_log_def(X_te_t).numpy()

y_pred_saint_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_saint(y_test, y_pred_saint_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)")

Model -> MAD: 2.1699 ha | MAE: 19.8465 ha | RMSE: 109.9740 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAINT Transformer,11.4767,24.4245,108.8721
1,2 - Normal Veri (Bayesyen Optuna HPO),SAINT Transformer,11.0679,24.1684,108.9126
2,3 - Log-Dönüşümlü Veri (Varsayılan),SAINT Transformer,2.1699,19.8465,109.9740


(np.float64(2.169944167137146),
 19.84648098001113,
 np.float64(109.97395932802436))

### Deney 4:

In [27]:
if len(saint_results) > 0 and saint_results[-1]["Deney Aşaması"] == "4 - Log Veri (Bayesyen Optuna HPO)":
    saint_results.pop()

def objective_saint_log_v2(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32])
    n_blk = trial.suggest_int("n_blocks", 1, 2)
    n_hds = trial.suggest_categorical("n_heads", [2, 4])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_blocks": n_blk,
        "n_heads": n_hds,
        "dropout": trial.suggest_float("dropout", 0.0, 0.25),
        "lr": trial.suggest_float("lr", 0.001, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    val_size = int(len(X_train_scaled) * 0.2)
    X_tr_inner = torch.tensor(X_train_scaled[:-val_size], dtype=torch.float32)
    y_tr_inner = torch.tensor(y_train_log[:-val_size], dtype=torch.float32).unsqueeze(1)
    X_val_inner = torch.tensor(X_train_scaled[-val_size:], dtype=torch.float32)
    y_val_inner_ha = y_train[-val_size:]
    
    ds_tr = TensorDataset(X_tr_inner, y_tr_inner)
    ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
    
    model = SAINTTabular(
        num_features=X_train_scaled.shape[1],
        d_token=params["d_token"],
        n_blocks=params["n_blocks"],
        n_heads=params["n_heads"],
        dropout=params["dropout"]
    )
    opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    crit = nn.MSELoss()
    
    model.train()
    for epoch in range(80):
        for bx, by in ld_tr:
            opt.zero_grad()
            p_b = model(bx)
            l = crit(p_b, by)
            l.backward()
            opt.step()
            
        if epoch in [25, 50]:
            model.eval()
            with torch.no_grad():
                preds_val_log = model(X_val_inner).numpy()
            val_mad = np.median(np.abs(y_val_inner_ha - np.clip(np.expm1(preds_val_log), 0, None)))
            trial.report(val_mad, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            model.train()
            
    model.eval()
    with torch.no_grad():
        preds_log_val = model(X_val_inner).numpy()
    return np.median(np.abs(y_val_inner_ha - np.clip(np.expm1(preds_log_val), 0, None)))

study_saint_log_v2 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=25)
)
study_saint_log_v2.optimize(objective_saint_log_v2, n_trials=12, show_progress_bar=False)
best_saint_p_v2 = study_saint_log_v2.best_params

if best_saint_p_v2["d_token"] % best_saint_p_v2["n_heads"] != 0:
    best_saint_p_v2["n_heads"] = 2

model_saint_opt_v2 = SAINTTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_saint_p_v2["d_token"],
    n_blocks=best_saint_p_v2["n_blocks"],
    n_heads=best_saint_p_v2["n_heads"],
    dropout=best_saint_p_v2["dropout"]
)
opt_saint_v2 = optim.AdamW(model_saint_opt_v2.parameters(), lr=best_saint_p_v2["lr"], weight_decay=best_saint_p_v2["weight_decay"])
scheduler_saint_v2 = optim.lr_scheduler.CosineAnnealingLR(opt_saint_v2, T_max=150)
crit_saint_v2 = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=64, shuffle=True)

model_saint_opt_v2.train()
for epoch in range(150):
    for bx, by in ld_full_tr_log:
        opt_saint_v2.zero_grad()
        p_b = model_saint_opt_v2(bx)
        l = crit_saint_v2(p_b, by)
        l.backward()
        opt_saint_v2.step()
    scheduler_saint_v2.step()

model_saint_opt_v2.eval()
with torch.no_grad():
    preds_log_opt_saint_v2 = model_saint_opt_v2(X_te_t).numpy()

y_pred_saint_opt_v2_ha = np.clip(np.expm1(preds_log_opt_saint_v2), 0, None)

evaluate_saint(y_test, y_pred_saint_opt_v2_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

Model -> MAD: 2.0566 ha | MAE: 19.8169 ha | RMSE: 109.9920 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAINT Transformer,11.4767,24.4245,108.8721
1,2 - Normal Veri (Bayesyen Optuna HPO),SAINT Transformer,11.0679,24.1684,108.9126
2,3 - Log-Dönüşümlü Veri (Varsayılan),SAINT Transformer,2.1699,19.8465,109.9740
3,4 - Log Veri (Bayesyen Optuna HPO),SAINT Transformer,2.0566,19.8169,109.9920


(np.float64(2.0566117763519287),
 19.81693028092384,
 np.float64(109.99204177107244))

## DANNet Deneyleri:

### Deney 1:

In [28]:
class AbstractLayer(nn.Module):
    def __init__(self, num_features, k_abstract=16, dropout=0.1):
        super().__init__()
        self.feature_weights = nn.Parameter(torch.randn(num_features, k_abstract) * 0.02)
        self.block_gate = nn.Sequential(
            nn.Linear(num_features, k_abstract),
            nn.BatchNorm1d(k_abstract),
            nn.Sigmoid()
        )
        self.proj = nn.Sequential(
            nn.Linear(k_abstract, k_abstract),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        gated = x.matmul(self.feature_weights) * self.block_gate(x)
        return self.proj(gated)

class DANNetTabular(nn.Module):
    def __init__(self, num_features, k_abstract=16, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([AbstractLayer(num_features if i==0 else k_abstract, k_abstract, dropout) for i in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(k_abstract),
            nn.ReLU(),
            nn.Linear(k_abstract, 1)
        )

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer(h)
        return self.head(h).squeeze(-1)

if 'dannet_results' not in globals() or len(dannet_results) > 4:
    dannet_results = []

def evaluate_dannet(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    dannet_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "DANNet Abstract Layer",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[DANNET ABSTRACT LAYER] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_dannet = pd.DataFrame(dannet_results)
    styled_table = (
        df_dannet.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#e65100'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#e65100'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_dannet_raw = TensorDataset(X_tr_t, y_tr_t)
loader_dannet_raw = DataLoader(dataset_dannet_raw, batch_size=32, shuffle=True)

model_dannet_def = DANNetTabular(num_features=X_train_scaled.shape[1], k_abstract=16, n_layers=2, dropout=0.1)
optimizer_dannet = optim.AdamW(model_dannet_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_dannet = nn.MSELoss()

model_dannet_def.train()
for epoch in range(150):
    for bx, by in loader_dannet_raw:
        optimizer_dannet.zero_grad()
        preds_b = model_dannet_def(bx)
        loss = criterion_dannet(preds_b, by)
        loss.backward()
        optimizer_dannet.step()

model_dannet_def.eval()
with torch.no_grad():
    y_pred_dannet_raw = model_dannet_def(X_te_t).numpy()

evaluate_dannet(y_test, y_pred_dannet_raw, "1 - Normal Veri (Varsayılan)")

[DANNET ABSTRACT LAYER] - 1 - NORMAL VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 11.5804 ha | MAE: 24.4746 ha | RMSE: 108.8715 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),DANNet Abstract Layer,11.5804,24.4746,108.8715


(np.float64(11.580384254455566),
 24.474561195006736,
 np.float64(108.87147551052286))

### Deney 2:

In [29]:
def objective_dannet_raw(trial):
    params = {
        "k_abstract": trial.suggest_categorical("k_abstract", [8, 16, 32]),
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.0, 0.25),
        "lr": trial.suggest_float("lr", 0.001, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    val_size = int(len(X_train_scaled) * 0.2)
    X_tr_inner = torch.tensor(X_train_scaled[:-val_size], dtype=torch.float32)
    y_tr_inner = torch.tensor(y_train[:-val_size], dtype=torch.float32).unsqueeze(1)
    X_val_inner = torch.tensor(X_train_scaled[-val_size:], dtype=torch.float32)
    y_val_inner = y_train[-val_size:]
    
    ds_tr = TensorDataset(X_tr_inner, y_tr_inner)
    ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
    
    model = DANNetTabular(
        num_features=X_train_scaled.shape[1],
        k_abstract=params["k_abstract"],
        n_layers=params["n_layers"],
        dropout=params["dropout"]
    )
    opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    crit = nn.MSELoss()
    
    model.train()
    for epoch in range(80):
        for bx, by in ld_tr:
            opt.zero_grad()
            p_b = model(bx)
            l = crit(p_b, by)
            l.backward()
            opt.step()
            
        if epoch in [25, 50]:
            model.eval()
            with torch.no_grad():
                preds_val = model(X_val_inner).numpy()
            val_mad = np.median(np.abs(y_val_inner - preds_val))
            trial.report(val_mad, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            model.train()
            
    model.eval()
    with torch.no_grad():
        preds_val = model(X_val_inner).numpy()
    return np.median(np.abs(y_val_inner - preds_val))

study_dannet_raw = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=25)
)
study_dannet_raw.optimize(objective_dannet_raw, n_trials=12, show_progress_bar=False)
best_dannet_p_raw = study_dannet_raw.best_params

model_dannet_opt_raw = DANNetTabular(
    num_features=X_train_scaled.shape[1],
    k_abstract=best_dannet_p_raw["k_abstract"],
    n_layers=best_dannet_p_raw["n_layers"],
    dropout=best_dannet_p_raw["dropout"]
)
opt_dannet_raw = optim.AdamW(model_dannet_opt_raw.parameters(), lr=best_dannet_p_raw["lr"], weight_decay=best_dannet_p_raw["weight_decay"])
scheduler_dannet_raw = optim.lr_scheduler.CosineAnnealingLR(opt_dannet_raw, T_max=150)
crit_dannet_raw = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=64, shuffle=True)

model_dannet_opt_raw.train()
for epoch in range(150):
    for bx, by in ld_full_tr:
        opt_dannet_raw.zero_grad()
        p_b = model_dannet_opt_raw(bx)
        l = crit_dannet_raw(p_b, by)
        l.backward()
        opt_dannet_raw.step()
    scheduler_dannet_raw.step()

model_dannet_opt_raw.eval()
with torch.no_grad():
    y_pred_dannet_opt_raw = model_dannet_opt_raw(X_te_t).numpy()

evaluate_dannet(y_test, y_pred_dannet_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)")

[DANNET ABSTRACT LAYER] - 2 - NORMAL VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 5.0093 ha | MAE: 20.8264 ha | RMSE: 109.5553 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),DANNet Abstract Layer,11.5804,24.4746,108.8715
1,2 - Normal Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,5.0093,20.8264,109.5553


(np.float64(5.009344577789307),
 20.826445644085222,
 np.float64(109.55527548099388))

### Deney 3:

In [30]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_dannet_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_dannet_log = DataLoader(dataset_dannet_log, batch_size=32, shuffle=True)

model_dannet_log_def = DANNetTabular(num_features=X_train_scaled.shape[1], k_abstract=16, n_layers=2, dropout=0.1)
optimizer_dannet_log = optim.AdamW(model_dannet_log_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_dannet_log = nn.MSELoss()

model_dannet_log_def.train()
for epoch in range(150):
    for bx, by in loader_dannet_log:
        optimizer_dannet_log.zero_grad()
        preds_b = model_dannet_log_def(bx)
        loss = criterion_dannet_log(preds_b, by)
        loss.backward()
        optimizer_dannet_log.step()

model_dannet_log_def.eval()
with torch.no_grad():
    preds_log_val = model_dannet_log_def(X_te_t).numpy()

y_pred_dannet_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_dannet(y_test, y_pred_dannet_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)")

[DANNET ABSTRACT LAYER] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 2.0726 ha | MAE: 19.8281 ha | RMSE: 109.9867 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),DANNet Abstract Layer,11.5804,24.4746,108.8715
1,2 - Normal Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,5.0093,20.8264,109.5553
2,3 - Log-Dönüşümlü Veri (Varsayılan),DANNet Abstract Layer,2.0726,19.8281,109.9867


(np.float64(2.072610020637512),
 19.828091156666094,
 np.float64(109.98674267492365))

### Deney 4:

In [31]:
def objective_dannet_log(trial):
    params = {
        "k_abstract": trial.suggest_categorical("k_abstract", [8, 16, 32]),
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.0, 0.25),
        "lr": trial.suggest_float("lr", 0.001, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    val_size = int(len(X_train_scaled) * 0.2)
    X_tr_inner = torch.tensor(X_train_scaled[:-val_size], dtype=torch.float32)
    y_tr_inner = torch.tensor(y_train_log[:-val_size], dtype=torch.float32).unsqueeze(1)
    X_val_inner = torch.tensor(X_train_scaled[-val_size:], dtype=torch.float32)
    y_val_inner_ha = y_train[-val_size:]
    
    ds_tr = TensorDataset(X_tr_inner, y_tr_inner)
    ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
    
    model = DANNetTabular(
        num_features=X_train_scaled.shape[1],
        k_abstract=params["k_abstract"],
        n_layers=params["n_layers"],
        dropout=params["dropout"]
    )
    opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    crit = nn.MSELoss()
    
    model.train()
    for epoch in range(80):
        for bx, by in ld_tr:
            opt.zero_grad()
            p_b = model(bx)
            l = crit(p_b, by)
            l.backward()
            opt.step()
            
        if epoch in [25, 50]:
            model.eval()
            with torch.no_grad():
                preds_val_log = model(X_val_inner).numpy()
            val_mad = np.median(np.abs(y_val_inner_ha - np.clip(np.expm1(preds_val_log), 0, None)))
            trial.report(val_mad, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            model.train()
            
    model.eval()
    with torch.no_grad():
        preds_log_val = model(X_val_inner).numpy()
    return np.median(np.abs(y_val_inner_ha - np.clip(np.expm1(preds_log_val), 0, None)))

study_dannet_log = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=25)
)
study_dannet_log.optimize(objective_dannet_log, n_trials=12, show_progress_bar=False)
best_dannet_p_log = study_dannet_log.best_params

model_dannet_opt_log = DANNetTabular(
    num_features=X_train_scaled.shape[1],
    k_abstract=best_dannet_p_log["k_abstract"],
    n_layers=best_dannet_p_log["n_layers"],
    dropout=best_dannet_p_log["dropout"]
)
opt_dannet_log = optim.AdamW(model_dannet_opt_log.parameters(), lr=best_dannet_p_log["lr"], weight_decay=best_dannet_p_log["weight_decay"])
scheduler_dannet_log = optim.lr_scheduler.CosineAnnealingLR(opt_dannet_log, T_max=150)
crit_dannet_log = nn.MSELoss()

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=64, shuffle=True)

model_dannet_opt_log.train()
for epoch in range(150):
    for bx, by in ld_full_tr_log:
        opt_dannet_log.zero_grad()
        p_b = model_dannet_opt_log(bx)
        l = crit_dannet_log(p_b, by)
        l.backward()
        opt_dannet_log.step()
    scheduler_dannet_log.step()

model_dannet_opt_log.eval()
with torch.no_grad():
    preds_log_opt_dannet = model_dannet_opt_log(X_te_t).numpy()

y_pred_dannet_opt_log_ha = np.clip(np.expm1(preds_log_opt_dannet), 0, None)

evaluate_dannet(y_test, y_pred_dannet_opt_log_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

[DANNET ABSTRACT LAYER] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 2.0941 ha | MAE: 19.8424 ha | RMSE: 109.9917 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),DANNet Abstract Layer,11.5804,24.4746,108.8715
1,2 - Normal Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,5.0093,20.8264,109.5553
2,3 - Log-Dönüşümlü Veri (Varsayılan),DANNet Abstract Layer,2.0726,19.8281,109.9867
3,4 - Log Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,2.0941,19.8424,109.9917


(np.float64(2.0940845012664795),
 19.842357287682017,
 np.float64(109.99170188732452))

In [32]:
if len(dannet_results) > 0 and dannet_results[-1]["Deney Aşaması"] == "4 - Log Veri (Bayesyen Optuna HPO)":
    dannet_results.pop()

class AbstractLayerV2(nn.Module):
    def __init__(self, num_features, k_abstract=16, dropout=0.1, tau=1.0):
        super().__init__()
        self.tau = tau
        self.feature_weights = nn.Parameter(torch.randn(num_features, k_abstract) * 0.02)
        self.block_gate = nn.Sequential(
            nn.Linear(num_features, k_abstract),
            nn.BatchNorm1d(k_abstract)
        )
        self.proj = nn.Sequential(
            nn.Linear(k_abstract, k_abstract),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        gate = torch.sigmoid(self.block_gate(x) / self.tau)
        gated = x.matmul(self.feature_weights) * gate
        return self.proj(gated)

class EnhancedDANNetTabular(nn.Module):
    def __init__(self, num_features, k_abstract=16, n_layers=2, dropout=0.1, tau=1.0):
        super().__init__()
        self.layers = nn.ModuleList([AbstractLayerV2(num_features if i==0 else k_abstract, k_abstract, dropout, tau) for i in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(k_abstract),
            nn.ReLU(),
            nn.Linear(k_abstract, 1)
        )

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer(h)
        return self.head(h).squeeze(-1)

def objective_dannet_log_v2(trial):
    params = {
        "k_abstract": trial.suggest_categorical("k_abstract", [16, 24, 32, 48, 64]),
        "n_layers": trial.suggest_int("n_layers", 1, 4),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "tau": trial.suggest_float("tau", 0.5, 2.0),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
        
        model = EnhancedDANNetTabular(
            num_features=X_train_scaled.shape[1],
            k_abstract=params["k_abstract"],
            n_layers=params["n_layers"],
            dropout=params["dropout"],
            tau=params["tau"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.HuberLoss(delta=1.0)
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
            if epoch == 40:
                model.eval()
                with torch.no_grad():
                    p_val_chk = model(X_val_t).numpy()
                mad_chk = np.median(np.abs(y_train[val_idx] - np.clip(np.expm1(p_val_chk), 0, None)))
                trial.report(mad_chk, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                model.train()
                
        model.eval()
        with torch.no_grad():
            preds_log_val = model(X_val_t).numpy()
        preds_ha_val = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_val)))
        
    return np.mean(cv_mads)

study_dannet_v2 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=40)
)
study_dannet_v2.optimize(objective_dannet_log_v2, n_trials=30, show_progress_bar=False)
best_dannet_p_v2 = study_dannet_v2.best_params

model_dannet_opt_v2 = EnhancedDANNetTabular(
    num_features=X_train_scaled.shape[1],
    k_abstract=best_dannet_p_v2["k_abstract"],
    n_layers=best_dannet_p_v2["n_layers"],
    dropout=best_dannet_p_v2["dropout"],
    tau=best_dannet_p_v2["tau"]
)
opt_dannet_v2 = optim.AdamW(model_dannet_opt_v2.parameters(), lr=best_dannet_p_v2["lr"], weight_decay=best_dannet_p_v2["weight_decay"])
scheduler_dannet_v2 = optim.lr_scheduler.CosineAnnealingLR(opt_dannet_v2, T_max=180)
crit_dannet_v2 = nn.HuberLoss(delta=1.0)

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=64, shuffle=True)

model_dannet_opt_v2.train()
for epoch in range(180):
    for bx, by in ld_full_tr_log:
        opt_dannet_v2.zero_grad()
        p_b = model_dannet_opt_v2(bx)
        l = crit_dannet_v2(p_b, by)
        l.backward()
        opt_dannet_v2.step()
    scheduler_dannet_v2.step()

model_dannet_opt_v2.eval()
with torch.no_grad():
    preds_log_opt_dannet_v2 = model_dannet_opt_v2(X_te_t).numpy()

y_pred_dannet_opt_v2_ha = np.clip(np.expm1(preds_log_opt_dannet_v2), 0, None)

evaluate_dannet(y_test, y_pred_dannet_opt_v2_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

[DANNET ABSTRACT LAYER] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 1.1779 ha | MAE: 19.6872 ha | RMSE: 110.1331 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),DANNet Abstract Layer,11.5804,24.4746,108.8715
1,2 - Normal Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,5.0093,20.8264,109.5553
2,3 - Log-Dönüşümlü Veri (Varsayılan),DANNet Abstract Layer,2.0726,19.8281,109.9867
3,4 - Log Veri (Bayesyen Optuna HPO),DANNet Abstract Layer,1.1779,19.6872,110.1331


(np.float64(1.17790949344635),
 19.687238935782357,
 np.float64(110.13312933510277))

## SAIL Deneyler:

### Deney 1:

In [33]:
class SparseAttentionLayer(nn.Module):
    def __init__(self, d_model, n_heads=4, sparsity_factor=0.5, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.sparsity_factor = sparsity_factor

    def forward(self, x):
        B, N, D = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        
        if self.sparsity_factor > 0.0 and N > 1:
            k_keep = max(1, int(N * (1.0 - self.sparsity_factor)))
            topk_vals, _ = torch.topk(scores, k=k_keep, dim=-1)
            threshold = topk_vals[..., -1:]
            scores = torch.where(scores >= threshold, scores, torch.tensor(-1e9, device=scores.device))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, D)
        return self.out_proj(out)

class SAILTabular(nn.Module):
    def __init__(self, num_features, d_token=32, n_layers=2, n_heads=4, sparsity=0.3, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_token)
        self.layers = nn.ModuleList([
            nn.Sequential(
                SparseAttentionLayer(d_token, n_heads, sparsity, dropout),
                nn.LayerNorm(d_token),
                nn.Linear(d_token, d_token * 2),
                nn.ReLU(),
                nn.Linear(d_token * 2, d_token),
                nn.LayerNorm(d_token)
            ) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_token, 1)

    def forward(self, x):
        h = self.tokenizer(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.head(h[:, 0, :]).squeeze(-1)

if 'sail_results' not in globals() or len(sail_results) > 4:
    sail_results = []

def evaluate_sail(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    sail_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "SAIL Sparse Attention",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[SAIL SPARSE ATTENTION] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_sail = pd.DataFrame(sail_results)
    styled_table = (
        df_sail.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Blues")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#0d47a1'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#0d47a1'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_sail_raw = TensorDataset(X_tr_t, y_tr_t)
loader_sail_raw = DataLoader(dataset_sail_raw, batch_size=32, shuffle=True)

model_sail_def = SAILTabular(num_features=X_train_scaled.shape[1], d_token=32, n_layers=2, n_heads=4, sparsity=0.3, dropout=0.1)
optimizer_sail = optim.AdamW(model_sail_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_sail = nn.MSELoss()

model_sail_def.train()
for epoch in range(150):
    for bx, by in loader_sail_raw:
        optimizer_sail.zero_grad()
        preds_b = model_sail_def(bx)
        loss = criterion_sail(preds_b, by)
        loss.backward()
        optimizer_sail.step()

model_sail_def.eval()
with torch.no_grad():
    y_pred_sail_raw = model_sail_def(X_te_t).numpy()

evaluate_sail(y_test, y_pred_sail_raw, "1 - Normal Veri (Varsayılan)")

[SAIL SPARSE ATTENTION] - 1 - NORMAL VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 11.0933 ha | MAE: 24.1560 ha | RMSE: 108.9088 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAIL Sparse Attention,11.0933,24.1560,108.9088


(np.float64(11.093315124511719),
 24.156025105989894,
 np.float64(108.90876992124731))

### Deney 2:

In [34]:
def objective_sail_raw(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32, 64])
    n_lay = trial.suggest_int("n_layers", 1, 3)
    n_hds = trial.suggest_categorical("n_heads", [2, 4, 8])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_layers": n_lay,
        "n_heads": n_hds,
        "sparsity": trial.suggest_float("sparsity", 0.1, 0.6),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
        
        model = SAILTabular(
            num_features=X_train_scaled.shape[1],
            d_token=params["d_token"],
            n_layers=params["n_layers"],
            n_heads=params["n_heads"],
            sparsity=params["sparsity"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.HuberLoss(delta=1.0)
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
            if epoch == 40:
                model.eval()
                with torch.no_grad():
                    preds_val_chk = model(X_val_t).numpy()
                mad_chk = np.median(np.abs(y_train[val_idx] - preds_val_chk))
                trial.report(mad_chk, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                model.train()
                
        model.eval()
        with torch.no_grad():
            preds_val = model(X_val_t).numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_val)))
        
    return np.mean(cv_mads)

study_sail_raw = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=35)
)
study_sail_raw.optimize(objective_sail_raw, n_trials=15, show_progress_bar=False)
best_sail_p_raw = study_sail_raw.best_params

if best_sail_p_raw["d_token"] % best_sail_p_raw["n_heads"] != 0:
    best_sail_p_raw["n_heads"] = 2

model_sail_opt_raw = SAILTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_sail_p_raw["d_token"],
    n_layers=best_sail_p_raw["n_layers"],
    n_heads=best_sail_p_raw["n_heads"],
    sparsity=best_sail_p_raw["sparsity"],
    dropout=best_sail_p_raw["dropout"]
)
opt_sail_raw = optim.AdamW(model_sail_opt_raw.parameters(), lr=best_sail_p_raw["lr"], weight_decay=best_sail_p_raw["weight_decay"])
scheduler_sail_raw = optim.lr_scheduler.CosineAnnealingLR(opt_sail_raw, T_max=150)
crit_sail_raw = nn.HuberLoss(delta=1.0)

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=64, shuffle=True)

model_sail_opt_raw.train()
for epoch in range(150):
    for bx, by in ld_full_tr:
        opt_sail_raw.zero_grad()
        p_b = model_sail_opt_raw(bx)
        l = crit_sail_raw(p_b, by)
        l.backward()
        opt_sail_raw.step()
    scheduler_sail_raw.step()

model_sail_opt_raw.eval()
with torch.no_grad():
    y_pred_sail_opt_raw = model_sail_opt_raw(X_te_t).numpy()

evaluate_sail(y_test, y_pred_sail_opt_raw, "2 - Normal Veri (Bayesyen Optuna HPO)")

[SAIL SPARSE ATTENTION] - 2 - NORMAL VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 0.9014 ha | MAE: 19.6639 ha | RMSE: 110.1799 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAIL Sparse Attention,11.0933,24.1560,108.9088
1,2 - Normal Veri (Bayesyen Optuna HPO),SAIL Sparse Attention,0.9014,19.6639,110.1799


(np.float64(0.9013960957527161),
 19.66394633022638,
 np.float64(110.17994074454587))

### Deney 3:

In [35]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_sail_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_sail_log = DataLoader(dataset_sail_log, batch_size=32, shuffle=True)

model_sail_log_def = SAILTabular(num_features=X_train_scaled.shape[1], d_token=32, n_layers=2, n_heads=4, sparsity=0.3, dropout=0.1)
optimizer_sail_log = optim.AdamW(model_sail_log_def.parameters(), lr=0.001, weight_decay=1e-4)
criterion_sail_log = nn.MSELoss()

model_sail_log_def.train()
for epoch in range(150):
    for bx, by in loader_sail_log:
        optimizer_sail_log.zero_grad()
        preds_b = model_sail_log_def(bx)
        loss = criterion_sail_log(preds_b, by)
        loss.backward()
        optimizer_sail_log.step()

model_sail_log_def.eval()
with torch.no_grad():
    preds_log_val = model_sail_log_def(X_te_t).numpy()

y_pred_sail_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_sail(y_test, y_pred_sail_log_def_ha, "3 - Log-Dönüşümlü Veri (Varsayılan)")

[SAIL SPARSE ATTENTION] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN) SONUÇLARI
Model -> MAD: 1.8399 ha | MAE: 19.7837 ha | RMSE: 110.0240 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAIL Sparse Attention,11.0933,24.1560,108.9088
1,2 - Normal Veri (Bayesyen Optuna HPO),SAIL Sparse Attention,0.9014,19.6639,110.1799
2,3 - Log-Dönüşümlü Veri (Varsayılan),SAIL Sparse Attention,1.8399,19.7837,110.0240


(np.float64(1.8398783206939697),
 19.783675108689526,
 np.float64(110.02398009085785))

### Deney 4:

In [36]:
def objective_sail_log(trial):
    d_tok = trial.suggest_categorical("d_token", [16, 32, 64])
    n_lay = trial.suggest_int("n_layers", 1, 3)
    n_hds = trial.suggest_categorical("n_heads", [2, 4, 8])
    if d_tok % n_hds != 0:
        n_hds = 2
        
    params = {
        "d_token": d_tok,
        "n_layers": n_lay,
        "n_heads": n_hds,
        "sparsity": trial.suggest_float("sparsity", 0.15, 0.65),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "lr": trial.suggest_float("lr", 0.0005, 0.008, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
        
        model = SAILTabular(
            num_features=X_train_scaled.shape[1],
            d_token=params["d_token"],
            n_layers=params["n_layers"],
            n_heads=params["n_heads"],
            sparsity=params["sparsity"],
            dropout=params["dropout"]
        )
        opt = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        crit = nn.HuberLoss(delta=1.0)
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                p_b = model(bx)
                l = crit(p_b, by)
                l.backward()
                opt.step()
                
            if epoch == 40:
                model.eval()
                with torch.no_grad():
                    preds_val_chk_log = model(X_val_t).numpy()
                mad_chk = np.median(np.abs(y_train[val_idx] - np.clip(np.expm1(preds_val_chk_log), 0, None)))
                trial.report(mad_chk, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                model.train()
                
        model.eval()
        with torch.no_grad():
            preds_log_val = model(X_val_t).numpy()
        preds_ha_val = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_val)))
        
    return np.mean(cv_mads)

study_sail_log = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=35)
)
study_sail_log.optimize(objective_sail_log, n_trials=30, show_progress_bar=False)
best_sail_p_log = study_sail_log.best_params

if best_sail_p_log["d_token"] % best_sail_p_log["n_heads"] != 0:
    best_sail_p_log["n_heads"] = 2

model_sail_opt_log = SAILTabular(
    num_features=X_train_scaled.shape[1],
    d_token=best_sail_p_log["d_token"],
    n_layers=best_sail_p_log["n_layers"],
    n_heads=best_sail_p_log["n_heads"],
    sparsity=best_sail_p_log["sparsity"],
    dropout=best_sail_p_log["dropout"]
)
opt_sail_log = optim.AdamW(model_sail_opt_log.parameters(), lr=best_sail_p_log["lr"], weight_decay=best_sail_p_log["weight_decay"])
scheduler_sail_log = optim.lr_scheduler.CosineAnnealingLR(opt_sail_log, T_max=160)
crit_sail_log = nn.HuberLoss(delta=1.0)

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=64, shuffle=True)

model_sail_opt_log.train()
for epoch in range(160):
    for bx, by in ld_full_tr_log:
        opt_sail_log.zero_grad()
        p_b = model_sail_opt_log(bx)
        l = crit_sail_log(p_b, by)
        l.backward()
        opt_sail_log.step()
    scheduler_sail_log.step()

model_sail_opt_log.eval()
with torch.no_grad():
    preds_log_opt_sail = model_sail_opt_log(X_te_t).numpy()

y_pred_sail_opt_log_ha = np.clip(np.expm1(preds_log_opt_sail), 0, None)

evaluate_sail(y_test, y_pred_sail_opt_log_ha, "4 - Log Veri (Bayesyen Optuna HPO)")

[SAIL SPARSE ATTENTION] - 4 - LOG VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 1.1391 ha | MAE: 19.6835 ha | RMSE: 110.1396 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan),SAIL Sparse Attention,11.0933,24.1560,108.9088
1,2 - Normal Veri (Bayesyen Optuna HPO),SAIL Sparse Attention,0.9014,19.6639,110.1799
2,3 - Log-Dönüşümlü Veri (Varsayılan),SAIL Sparse Attention,1.8399,19.7837,110.0240
3,4 - Log Veri (Bayesyen Optuna HPO),SAIL Sparse Attention,1.1391,19.6835,110.1396


(np.float64(1.1390557289123535),
 19.683483840869023,
 np.float64(110.13960592977284))